In [3]:
import pandas as pd
path = "data/dados.xlsx"
df = pd.read_excel(path)

In [4]:
df["descricao"].unique()

<ArrowStringArray>
[                        'Transação referente à contratação de pacote premium com acesso ampliado a recursos avançados, relatórios detalhados, integração com sistemas externos e suporte técnico dedicado durante todo o ciclo contratado.',
      'Cobrança vinculada à utilização de serviços recorrentes prestados ao cliente, contemplando manutenção da infraestrutura, monitoramento de desempenho e atendimento especializado conforme contrato vigente estabelecido previamente.',
                       'Pagamento referente à assinatura mensal de serviços digitais contratados pela empresa, incluindo acesso à plataforma, suporte técnico contínuo e atualizações periódicas com novos recursos e melhorias no sistema.',
                                                                                                                                                                                                                      'Pagamento de serviço',
             'Pagamento efetu

In [1]:
import requests

url = "http://localhost:11434/api/generate"

payload = {
    "model": "mistral",
    "prompt": "Explique o que é análise de dados em poucas palavras",
    "stream": False
}

response = requests.post(url, json=payload)

print(response.json()["response"])

 Análise de dados é o processo de explorar e interpretar informações contidas em conjuntos de dados para extrair insights, apoiando decisões e aplicativos práticos. Ele envolve limpeza, preparação, análise estatística e visualização dos dados.


In [2]:
from app.core.llm import llm_client

llm_client.initialize()

llm = llm_client.get_llm()

response = llm.complete("Explique o que é inadimplência em dados financeiros")

print(response.text)

 A inadimplência em dados financeiros se refere à situação em que uma entidade, seja ela uma empresa, um particular ou um governo, não consegue pagar suas obrigações financeiras a tempo. Isso pode incluir atrasos em pagamentos de juros, impostos, salários ou divídendos, bem como a falta de capacidade de financiar novas obrigações financeiras.

A inadimplência em dados financeiros pode ter implicações significativas para uma entidade, incluindo danos à sua reputação e possíveis penalidades legais ou financeiras. Pode também afetar a capacidade da entidade de obter crédito no futuro, pois os credores podem se tornar mais cautelosos ao fornecer empréstimos ou outras formas de financiamento àqueles com histórico de inadimplência.

Além disso, a inadimplência pode ser um sinal de problemas financeiros mais amplos, como uma fraqueza na gestão da empresa ou uma crise econômica em curso. Portanto, é importante que as entidades identifiquem e corrijam qualquer problema de inadimplência o mais r

In [6]:
list_desc = df["descricao"].unique()
list_desc

<ArrowStringArray>
[                        'Transação referente à contratação de pacote premium com acesso ampliado a recursos avançados, relatórios detalhados, integração com sistemas externos e suporte técnico dedicado durante todo o ciclo contratado.',
      'Cobrança vinculada à utilização de serviços recorrentes prestados ao cliente, contemplando manutenção da infraestrutura, monitoramento de desempenho e atendimento especializado conforme contrato vigente estabelecido previamente.',
                       'Pagamento referente à assinatura mensal de serviços digitais contratados pela empresa, incluindo acesso à plataforma, suporte técnico contínuo e atualizações periódicas com novos recursos e melhorias no sistema.',
                                                                                                                                                                                                                      'Pagamento de serviço',
             'Pagamento efetu

In [7]:
list_desc = df["descricao"].unique()

prompt_base = f"""
Você é um analista financeiro especializado em categorização de transações.

Você receberá descrições de transações e deve:

1. Criar categorias úteis para análise financeira
2. Definir possíveis valores para cada categoria
3. Ser consistente e evitar redundância
4. Pensar em categorias que possam ser aplicadas em um DataFrame

REGRAS IMPORTANTES:
- Retorne APENAS JSON válido
- Não explique nada
- Limite a no máximo 5 categorias
- Cada categoria deve ter no máximo 6 valores possíveis

DESCRIÇÕES:
{list_desc[:50]}

FORMATO DE SAÍDA:
{{
  "categorias": {{
    "nome_categoria": ["valor1", "valor2"]
  }}
}}
"""

In [24]:
list_desc = list(df["descricao"].unique())

pprompt_base = f"""
Você é um analista de financeiro especializado na categorização de transações.

Você receberá descrições de transações e deve conseguir indetificar caracteristicas comuns,
que possa ser representado de forma tabular. Serão categorias com valores limitados.

[EXEMPLO]
[
Categoria: "Recorrente"
Valores: ["True", "False"],

Categoria: "Tipo de recorrencia"
Valores: ["Mensal", "Anual", "Único"]
]
[FIM DO EXEMPLO]

REGRAS IMPORTANTES:
- Retorne APENAS JSON válido
- Não explique nada
- Limite a no máximo 5 categorias
- Cada categoria deve ter no máximo 6 valores possíveis
- As categoras podem ter no máximo 20 caracteres de comprimento
- Os valores podem no máximo ter 15 caracteres de comprimento
- Os nomes da categoria não podem incluir espaço. Invez disso utilize: "_"
- Os nomes dos valores não podem incluir espaço. Invez disso utilize: "_"
- seja sucinto ao nomear tanto categoria e valor, pois serão usados em dataframes


NÂO COLOQUE NOMES LONGOS

DESCRIÇÕES:

- {",\n\n -".join(list_desc)}

FORMATO DE SAÍDA:
{{
  "categorias": {{
    "nome_categoria": ["valor1", "valor2"]
  }}
}}
"""


In [26]:
from pydantic import BaseModel, Field
from typing import Dict, List

class CategorySchema(BaseModel):
    categorias: Dict[str, List[str]]

def validate_schema(schema: CategorySchema):
    categorias = schema.categorias

    # regra 1: máximo 5 categorias
    if len(categorias) > 5:
        raise ValueError("Mais de 5 categorias")

    for nome, valores in categorias.items():

        # regra 2: nome da categoria <= 20 chars
        if len(nome) > 25:
            raise ValueError(f"Categoria muito longa: {nome}")

        # regra 3: máximo 6 valores
        if len(valores) > 6:
            raise ValueError(f"Muitos valores em {nome}")

        for v in valores:
            # regra 4: valor <= 15 chars
            if len(v) > 20:
                raise ValueError(f"Valor muito longo: {v}")

    return True

In [27]:
import json
from app.core.llm import llm_client

llm_client.initialize()
llm = llm_client.get_llm()


def generate_schema(prompt_base, max_retries=3):
    last_error = None
    last_output = None

    for attempt in range(max_retries):

        # 🔹 1. monta prompt (com feedback do erro se existir)
        if last_error:
            prompt = f"""
Você gerou um JSON inválido anteriormente.

ERRO:
{last_error}

SAÍDA ANTERIOR:
{last_output}

Corrija o JSON mantendo as regras originais.

{prompt_base}
"""
        else:
            prompt = prompt_base

        # 🔹 2. chama LLM
        response = llm.complete(prompt)
        last_output = response.text

        try:
            # 🔹 3. parse JSON
            raw = json.loads(response.text)

            # 🔹 4. valida estrutura
            schema = CategorySchema(**raw)

            # 🔹 5. valida regras de negócio
            validate_schema(schema)

            return schema  # sucesso

        except Exception as e:
            last_error = str(e)

            print(f"[retry {attempt+1}] erro: {last_error}")

    raise ValueError("LLM falhou após múltiplas tentativas")

generate_schema(prompt_base)

[retry 1] erro: Categoria muito longa: Manutenção de infraestrutura
[retry 2] erro: Categoria muito longa: Recursos avançados e suporte técnico
[retry 3] erro: Expecting value: line 1 column 2 (char 1)


ValueError: LLM falhou após múltiplas tentativas

In [35]:
list_desc = list(df["descricao"].unique())

prompt_base = f"""
Você é um analista financeiro.

Sua tarefa é identificar possíveis TIPOS DE RECORRÊNCIA em transações financeiras.

Recorrência significa: presença de periodicidade na transação.

Exemplos:
- mensal
- anual
- semanal
- único
- recorrente

REGRAS:
- Analise as descrições abaixo
- Extraia apenas tipos de recorrência observados
- Não invente categorias novas fora desse contexto
- Use palavras curtas e padronizadas
- Use "_" no lugar de espaços
- Retorne APENAS JSON válido

DESCRIÇÕES:
{",\n - ".join(list_desc[:50])}

FORMATO DE SAÍDA:
{{
  "recorrencia": ["mensal", "anual", "unico", "recorrente"]
}}
"""

In [37]:
from pydantic import BaseModel
from typing import List

class RecurrenceSchema(BaseModel):
    recorrencia: List[str]


def validate_schema(schema: RecurrenceSchema):

    if len(schema.recorrencia) > 10:
        raise ValueError("Muitos valores de recorrência")

    for v in schema.recorrencia:
        if len(v) > 20:
            raise ValueError(f"Valor muito longo: {v}")

    return True

In [38]:
import json
from app.core.llm import llm_client

llm_client.initialize()
llm = llm_client.get_llm()


def generate_recurrence(prompt_base, max_retries=3):

    last_error = None
    last_output = None

    for attempt in range(max_retries):

        if last_error:
            prompt = f"""
Você gerou uma resposta inválida.

ERRO:
{last_error}

RESPOSTA ANTERIOR:
{last_output}

Corrija mantendo o formato JSON.

{prompt_base}
"""
        else:
            prompt = prompt_base

        response = llm.complete(prompt)
        last_output = response.text

        try:
            raw = json.loads(response.text)

            schema = RecurrenceSchema(**raw)

            validate_schema(schema)

            return schema

        except Exception as e:
            last_error = str(e)
            print(f"[retry {attempt+1}] {last_error}")

    raise ValueError("Falha ao gerar recorrência")

generate_recurrence(prompt_base)

RecurrenceSchema(recorrencia=['mensal', 'anual', 'unico', 'recorrente'])